# 01 — Data Collection

This notebook collects the two real data sources for the NPO recommender system:

1. **Content catalogue** from the NPO POMS API
2. **Observation data** from NPO Start's 'Aanbevolen voor jou' row

Run this notebook first before any other notebooks or the Streamlit app.


In [ ]:
import sys
sys.path.append('..')

import requests
import pandas as pd
import json
from pathlib import Path

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Directories ready.')

## Part 1 — POMS API Catalogue Pull

The POMS API is public and requires no authentication.
We collect broadcaster-labelled content items for all 6 NPO member broadcasters.


In [ ]:
BROADCASTERS = ['VPRO', 'NTR', 'MAX', 'AVROTROS', 'EO', 'KRO-NCRV']
POMS_BASE = 'https://api.poms.omroep.nl/media'

def fetch_broadcaster(broadcaster, max_results=100):
    items = []
    offset = 0
    while len(items) < max_results:
        params = {
            'broadcaster': broadcaster,
            'type': 'BROADCAST,SERIES',
            'max': 50,
            'offset': offset,
        }
        r = requests.get(POMS_BASE, params=params, timeout=15)
        if r.status_code != 200:
            print(f'Error {r.status_code} for {broadcaster}')
            break
        data = r.json()
        batch = data.get('items', [])
        if not batch:
            break
        items.extend(batch)
        offset += len(batch)
        if offset >= data.get('total', 0):
            break
    return items[:max_results]


all_items = []
for b in BROADCASTERS:
    print(f'Fetching {b}...')
    items = fetch_broadcaster(b, max_results=80)
    for item in items:
        genres = [g['value'] if isinstance(g, dict) else g for g in item.get('genres', [])]
        all_items.append({
            'item_id': item.get('mid', ''),
            'title': item.get('title', ''),
            'broadcaster': b,
            'genres': ','.join(genres),
            'description': item.get('shortDescription', ''),
            'publication_date': item.get('publishStart', ''),
        })
    print(f'  → {len(items)} items collected')

catalogue_df = pd.DataFrame(all_items)
catalogue_df.to_csv(PROCESSED_DIR / 'catalogue.csv', index=False)
print(f'\nTotal: {len(catalogue_df)} items saved to data/processed/catalogue.csv')
catalogue_df.head()

## Part 2 — Observation Sampling

Record which broadcasters appear in NPO Start's 'Aanbevolen voor jou' row.

**Manual steps:**
1. Open NPO Start in a private/incognito browser window (anonymous session)
2. Scroll to 'Aanbevolen voor jou'
3. Record the broadcaster for each visible item in the cell below
4. Repeat across 10–15 different anonymous sessions

This gives the baseline `rec_share` per broadcaster before our intervention.


In [ ]:
# Enter your observation data here.
# Each session: list the broadcasters of items visible in 'Aanbevolen voor jou'
# Format: session_id, broadcaster

observations = [
    # Session 1 — replace these with your actual observations
    {'session_id': 1, 'broadcaster': 'AVROTROS'},
    {'session_id': 1, 'broadcaster': 'AVROTROS'},
    {'session_id': 1, 'broadcaster': 'MAX'},
    {'session_id': 1, 'broadcaster': 'AVROTROS'},
    {'session_id': 1, 'broadcaster': 'KRO-NCRV'},
    # Add more sessions here...
]

obs_df = pd.DataFrame(observations)
obs_df.to_csv(RAW_DIR / 'observations.csv', index=False)

# Compute baseline rec_share
rec_share = obs_df['broadcaster'].value_counts(normalize=True)
print('Baseline rec_share per broadcaster:')
print(rec_share.to_string())

## Part 3 — Catalogue Share


In [ ]:
cat_share = catalogue_df['broadcaster'].value_counts(normalize=True)
print('Catalogue share per broadcaster:')
print(cat_share.to_string())

# Compute baseline Exposure Gap
import numpy as np
broadcasters = set(cat_share.index) | set(rec_share.index)
eg = np.mean([abs(rec_share.get(b, 0) - cat_share.get(b, 0)) for b in broadcasters])
print(f'\nBaseline Exposure Gap (EG): {eg:.4f}')